# Data Abstraction with AbstractionReviewer

This notebook demonstrates how to use `AbstractionReviewer` to extract structured data from article abstracts.

The `AbstractionReviewer` dynamically creates a Pydantic output model based on the `abstraction_keys` you define, allowing flexible extraction of any structured fields from text.

## Setup and Sample Data

In [ ]:
from lattereview.agentic import AbstractionReviewer
from pydantic_ai.models.test import TestModel

# Sample article abstracts
sample_items = [
    "We conducted a randomized controlled trial of 1,200 patients across 8 hospitals to evaluate "
    "an AI-assisted diagnostic tool for pneumonia detection on chest X-rays. The AI group showed "
    "a 15% reduction in diagnostic errors compared to standard care (p<0.001).",

    "A retrospective cohort study analyzed 5,400 electronic health records to predict 30-day "
    "hospital readmission using gradient boosted trees. The model achieved an AUC of 0.82 "
    "on the held-out test set, outperforming the LACE index (AUC 0.71).",

    "This cross-sectional survey of 320 radiologists examined attitudes toward AI adoption. "
    "67% reported willingness to use AI tools, while 45% expressed concerns about liability. "
    "Years of experience was negatively correlated with AI acceptance (r=-0.34, p<0.01).",
]

## Define Abstraction Schema

Use `abstraction_keys` to define the fields to extract and their types. Use `key_descriptions` to provide guidance on what each field should contain.

In [ ]:
abstractor = AbstractionReviewer(
    name="DataExtractor",
    backstory="You are an expert systematic reviewer who extracts structured data from medical research articles.",
    model=TestModel(),
    abstraction_keys={
        "study_design": str,
        "sample_size": int,
        "main_finding": str,
    },
    key_descriptions={
        "study_design": "The type of study (e.g., RCT, cohort, cross-sectional, case-control, systematic review)",
        "sample_size": "The number of participants, patients, or records analyzed",
        "main_finding": "A one-sentence summary of the primary result",
    },
    max_iterations=1,
)

print(f"Reviewer: {abstractor.name}")
print(f"Abstraction fields: {list(abstractor.abstraction_keys.keys())}")

## Extract Data from a Single Article

> **Note:** Uses `TestModel` for demonstration. For real extraction, use `model="openai:gpt-4o"` (requires `OPENAI_API_KEY`).

In [ ]:
result, cost = await abstractor.review_item(sample_items[0])

print("Extracted data:")
for key, value in result.items():
    print(f"  {key}: {value}")
print(f"\nCost: ${cost:.6f}")

## Extract Data from Multiple Articles

In [ ]:
results, total_cost = await abstractor.review_items(sample_items)

for i, r in enumerate(results):
    print(f"\nArticle {i+1}:")
    for key, value in r.items():
        print(f"  {key}: {value}")

print(f"\nTotal cost: ${total_cost:.6f}")

## The Dynamic Output Model

`AbstractionReviewer` builds a Pydantic model at runtime from `abstraction_keys`. The result dict mirrors these fields directly. You can define any combination of `str`, `int`, `float`, `bool`, or `list` types.

In [ ]:
# You can convert results to a DataFrame for analysis
import pandas as pd

results, _ = await abstractor.review_items(sample_items)
df = pd.DataFrame(results)
print(df.to_string(index=False))

## Using a Real Model

> **Requires API key:** Set `OPENAI_API_KEY` in your environment before running.

In [ ]:
# Uncomment to run with a real model:

# real_abstractor = AbstractionReviewer(
#     name="DataExtractor",
#     backstory="You are an expert systematic reviewer.",
#     model="openai:gpt-4o",
#     abstraction_keys={"study_design": str, "sample_size": int, "main_finding": str},
#     key_descriptions={
#         "study_design": "Type of study (RCT, cohort, etc.)",
#         "sample_size": "Number of participants or records",
#         "main_finding": "One-sentence primary result",
#     },
#     max_iterations=1,
# )
# result, cost = await real_abstractor.review_item(sample_items[0])
# print(result)